# RVC Kaggle 训练入口

这个 notebook 会 clone 你的 RVC 仓库，并使用仓库内的 `rvc_training_data/kaggle/train_rvc_kaggle.py` 执行完整训练流程：解压 Kaggle Dataset、预处理、F0、HuBERT 特征、训练、index 构建和导出。

使用前请在 Kaggle Notebook 右侧开启 GPU 和 Internet，并把本地生成的 30-60 分钟、单条 5-15 秒数据集 zip 添加为 Kaggle Dataset。

## 参数

`DATASET_ZIP` 指向 `/kaggle/input/...` 里的 zip。如果你添加的是已解压目录，可以把 `DATASET_ZIP = None`，然后设置 `DATASET_DIR`。

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/rikaaa0928/Retrieval-based-Voice-Conversion-WebUI.git'
REPO_DIR = Path('/kaggle/working/RVC')
VENV_DIR = Path('/kaggle/working/rvc_venv')
PYTHON = VENV_DIR / 'bin' / 'python'
UV = Path.home() / '.local' / 'bin' / 'uv'
EXPERIMENT = 'leijun_zh_v2_48k'

# zip 示例: /kaggle/input/zh-leijun-30m/zh_leijun_30m.zip
DATASET_ZIP = Path('/kaggle/input/zh-leijun-30m/zh_leijun_30m.zip')
DATASET_DIR = None
EXTRACT_DIR = Path('/kaggle/working/rvc_datasets')

SAMPLE_RATE = '48k'
VERSION = 'v2'
IF_F0 = 1
F0_METHOD = 'rmvpe'
GPUS = '0'
BATCH_SIZE = 8
TOTAL_EPOCH = 200
SAVE_EVERY_EPOCH = 20
SAVE_EVERY_WEIGHTS = 0

EXPORT_DIR = Path('/kaggle/working/rvc_models') / EXPERIMENT
PACKAGE_PATH = EXPORT_DIR.with_suffix('.zip')

## 检查 GPU

In [ ]:
!nvidia-smi

## 安装 RVC

Kaggle 需要开启 Internet 才能 clone 仓库、安装 pip 依赖和下载预训练模型。Notebook 会安装 `uv`，用 `uv venv` 创建隔离的 `/kaggle/working/rvc_venv`，再用 `uv pip` 安装训练依赖；这能避开 Kaggle 的 `venv/ensurepip` 问题，也不会混用系统 protobuf。训练结束后脚本会默认删除这个 venv 来节省 Output 空间。

In [ ]:
import os
import subprocess
import sys

os.chdir('/kaggle/working')
if REPO_DIR.exists():
    print(f'Reusing existing repo: {REPO_DIR}')
else:
    !git clone "$REPO_URL" "$REPO_DIR"

os.chdir(REPO_DIR)
print(f'Working directory: {Path.cwd()}')

requirements_file = 'rvc_training_data/kaggle/requirements-kaggle.txt'
if not UV.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--user', '--upgrade', 'uv'], check=True)

if not PYTHON.exists():
    subprocess.run([str(UV), 'venv', str(VENV_DIR)], check=True)

subprocess.run([str(UV), 'pip', 'install', '--python', str(PYTHON), '-r', requirements_file], check=True)
subprocess.run([str(PYTHON), '-c', 'import fairseq; print(f"fairseq import OK: {fairseq.__file__}")'], check=True)
subprocess.run([str(PYTHON), '-c', 'import pyworld; print(f"pyworld import OK: {pyworld.__file__}")'], check=True)

## 检查输入数据

In [ ]:
if DATASET_ZIP is not None:
    if not DATASET_ZIP.exists():
        raise FileNotFoundError(f'找不到数据 zip: {DATASET_ZIP}')
    print(f'Dataset zip: {DATASET_ZIP}')
elif DATASET_DIR is not None:
    DATASET_DIR = Path(DATASET_DIR)
    if not DATASET_DIR.exists():
        raise FileNotFoundError(f'找不到数据目录: {DATASET_DIR}')
    print(f'Dataset dir: {DATASET_DIR}')
else:
    raise ValueError('请设置 DATASET_ZIP 或 DATASET_DIR')

## 开始训练

In [ ]:
import os
import subprocess
import sys

SCRIPT_TARGET = REPO_DIR / 'rvc_training_data' / 'kaggle' / 'train_rvc_kaggle.py'
if not SCRIPT_TARGET.exists():
    raise FileNotFoundError(f'仓库中缺少训练脚本: {SCRIPT_TARGET}')

train_cmd = [
    str(PYTHON),
    str(SCRIPT_TARGET),
    '--repo-dir', str(REPO_DIR),
    '--experiment', EXPERIMENT,
    '--sample-rate', SAMPLE_RATE,
    '--version', VERSION,
    '--if-f0', str(IF_F0),
    '--f0-method', F0_METHOD,
    '--gpus', GPUS,
    '--batch-size', str(BATCH_SIZE),
    '--total-epoch', str(TOTAL_EPOCH),
    '--save-every-epoch', str(SAVE_EVERY_EPOCH),
    '--save-latest', '1',
    '--cache-gpu', '0',
    '--save-every-weights', str(SAVE_EVERY_WEIGHTS),
    '--extract-dir', str(EXTRACT_DIR),
    '--export-dir', str(EXPORT_DIR),
]

if DATASET_ZIP is not None:
    train_cmd.extend(['--dataset-zip', str(DATASET_ZIP)])
else:
    train_cmd.extend(['--dataset-dir', str(DATASET_DIR)])

print(' '.join(train_cmd))
train_env = os.environ.copy()
train_env['RVC_KAGGLE_PYTHON'] = str(PYTHON)
subprocess.run(train_cmd, cwd=REPO_DIR, check=True, env=train_env)

## 输出文件

Kaggle 会把 `/kaggle/working` 下的文件保存为 Notebook Output。默认只保留 `rvc_models/<实验名>.zip`，里面包含最终 `.pth`、`.index`、`train.log` 和 summary。

In [ ]:
if not PACKAGE_PATH.exists():
    raise FileNotFoundError(f'没有找到打包文件: {PACKAGE_PATH}')

print(f'Package: {PACKAGE_PATH}')
print(f'Size MB: {PACKAGE_PATH.stat().st_size / 1024 / 1024:.1f}')
if EXPORT_DIR.exists():
    print(f'Unpacked export dir: {EXPORT_DIR}')
    for path in sorted(EXPORT_DIR.iterdir()):
        print(path)